In [2]:
import torch
from IPython import display
from d2l import torch as d2l

batch_size = 256
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size)

In [4]:
num_inputs , num_outputs, num_hiddens = 784, 10, 256

W1 = nn.Parameter(torch.randn(num_inputs, num_hiddens, requires_grad=True))
b1 = nn.Parameter(torch.zeros(num_hiddens, requires_grad=True))
W2 = nn.Parameter(torch.randn(num_hiddens, num_outputs, requires_grad=True))
b2 = nn.Parameter(torch.zeros(num_outputs, requires_grad=True))

params = [W1, b1, W2, b2]

In [6]:
def relu(X):
    a = torch.zeros_like(X)
    return torch.where(X > 0, X, a)

def net(X):
    X = X.reshape((-1, num_inputs))
    H = relu(X@W1 + b1)
    return H@W2 + b2

loss = nn.CrossEntropyLoss()

In [14]:
# helper methods
def train_epoch_ch3(net, train_iter, loss, update):
    if isinstance(net, torch.nn.Module):
        net.train()
    metric = Accumulator(3)
    for X, y in train_iter:
        y_hat= net(X)
        l = loss(y_hat, y)
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            updater.step()
            metric.add(float(l) * len(y) , accuracy(y_hat,y), y.size().numel())
        else:
            l.sum().backward()
            updater(X.shape[0])
            metric.add(float(l.sum()), accuracy(y_hat, y), y.numel())
    return metric[0] / metric[2],  metric[1] / metric[2]

def train_ch3(net, train_iter, test_iter, loss, num_epochs, updater):
    # display with out animator
    for epoch in range(num_epochs):
        train_metrics = train_epoch_ch3(net, train_iter, loss, updater)
        test_acc = evaluate_accuracy(net, test_iter)
        print(f'epoch {epoch + 1}, train loss: {train_metrics[0]:.3f}, train acc: {train_metrics[1]:.3f}, test acc: {test_acc:.3f}')
        # animator.add(epoch + 1, train_metrics + (test_acc,))
    train_loss, train_acc = train_metrics


class Accumulator:
    def __init__(self, n):
        self.data = [0.0] * n # this will create n'th element in array [0.0, 0.0]
    def add(self, *args):
        self.data = [a + float(b) for a, b in zip(self.data, args)] # here is a simple function that add numbers to the diff element that created in data

    def reset(self):
        sekf.data = [0.0] * len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def accuracy(y_hat, y):
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())

def evaluate_accuracy(net, data_iter):
    if isinstance(net, torch.nn.Module):
        net.eval()
    metric = Accumulator(2)
    for X, y in data_iter:
        metric.add(accuracy(net(X), y), y.numel())
    return metric[0] / metric[1]



In [15]:
num_epochs, lr = 10, 0.1
updater = torch.optim.SGD(params, lr=lr)
train_ch3(net, train_iter, test_iter, loss, num_epochs, updater)

epoch 1, train loss: 2.271, train acc: 0.753, test acc: 0.745
epoch 2, train loss: 1.529, train acc: 0.766, test acc: 0.758
epoch 3, train loss: 1.214, train acc: 0.775, test acc: 0.757
epoch 4, train loss: 1.042, train acc: 0.782, test acc: 0.770
epoch 5, train loss: 0.929, train acc: 0.787, test acc: 0.771
epoch 6, train loss: 0.854, train acc: 0.790, test acc: 0.777
epoch 7, train loss: 0.794, train acc: 0.796, test acc: 0.781
epoch 8, train loss: 0.750, train acc: 0.798, test acc: 0.781
epoch 9, train loss: 0.717, train acc: 0.802, test acc: 0.783
epoch 10, train loss: 0.688, train acc: 0.804, test acc: 0.775


In [ ]:
# TODO: weight decay